# Учебный проект: AI агент для сравнения результатов обучения моделей предсказания выживаемости по данным Titanic

## Импорты
 1. Json - встроенный модуль, превращающий словари в строки и обратно
 2. matplotlib.use("Agg") - что бы окно с графиком не прерывало цикл агента. При обычном режиме plt пытается открыть окно и блокирует программу, а агент должен идти дальше
 3. plt - библиотека графиков
 4. np - массивы и числовые операции
 5. Pandas - иструмент для работы с таблицами, он же pd
 6. ollama - легкая локальная LLM

Библиотека Scikit-learn и ее импорты sklearn. :
 1. train_test_split, learning_curve - для разделения данных на обучающую и тестовую выборку, что бы модели не заучивали результаты и не завышали метрики. learning_curve - так же кривая обучения
 2. clone - создает чистую копию модели с теми же параметрами, что и у исходного ( защита от "грязных" общих моделей)
 3. from linear_model import LogisticRegression - Логистическая регрессия, модель классификации, предсказывает класс 0 или 1 считая вероятность
 4. from ensemble import RandomForestClassifier -  "случайный лес", строит анасамбль деревьев, а затем обьединает их прогнозы тем самым повышая точность предсказаний и снижая переобучение
 5. accuracy_score, ConfusionMatrixDisplay - метрика качества и матрица ошибок

In [ ]:
import json
import matplotlib
matplotlib.use('Agg') 
import ollama
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.base import clone
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

## Настройки и данные

In [3]:
LLM_MODEL = 'qwen2.5:3b' # Модель.
DF = pd.read_csv('train.csv') # Чтение данных Titanic для обучения и тестов
FEATURES = [
    'Pclass', # класс билета
    'Sex', # пол
    'Age', # возраст
    'Fare' # цена
] # признаки, т.е колонки из данных по которым предсказываем

MODELS ={
    'logreg' : LogisticRegression(max_iter=1000),
    'forest' : RandomForestClassifier(n_estimators=100, random_state=42),
}# модели для агента. Агент пришлёт строку "forest", и по ней мы получим нужную модель

#### LogisticRegression
Алгоритм машинного обучения с учителем, используемый для решения задач классификации. В отличие от линейной регрессии, которая предсказывает непрерывные значения, логистическая регрессия предсказывает вероятность того, что входные данные принадлежат к определенному классу. Используется для бинарной классификации, где на выходе может быть одна из двух возможных категорий, например «Да/Нет», «Истина/Ложь» или «0/1».
Для преобразования входных данных в значение вероятности от 0 до 1 используется сигмоидальная функция

Получает вероятность выжить от 0 до 1. Если вероятность больше 0,5, предсказывает «выжил».

 * max_iter=1000: максимальное число итераций оптимизатора. Веса подбираются пошагово, и по умолчанию шагов 100. Часто этого мало, и появляется предупреждение «не сошлось». 1000 даёт запас.

Параметры, которые мы не задавали, но стоит знать:
 * C=1.0: сила регуляризации (штрафа за слишком большие веса), чем меньше C, тем штраф сильнее. Он защищает от переобучения.
 * solver="lbfgs": алгоритм оптимизации.

#### RandomForestClassifier
Алгоритм ансамблевого машинного обучения, который строит несколько деревьев решений и объединяет их прогнозы для повышения точности и снижения переобучения. В Scikit‑learn классификатор «случайный лес» широко используется для решения задач классификации, поскольку он хорошо справляется с большими наборами данных и нелинейными взаимосвязями.

Cтроится 100 решающих деревьев. Каждое учится на случайной выборке строк и смотрит на случайный набор признаков, затем деревья голосуют, побеждает большинство. Разнообразие деревьев делает лес устойчивее одного дерева.